In [ ]:
# def predict_status(text):
#     # Clean the input text
#     cleaned_input = clean_text(text)

#     # Predict (returns 0 or 1)
#     prediction = model_pipeline.predict([cleaned_input])[0]

#     # Get Probability (confidence)
#     prob = model_pipeline.predict_proba([cleaned_input])[0][1]

#     # LOGIC: 0 = Clean, 1 = Hate
#     if prediction == 1:
#         label = "Hate/Toxic"
#         # ANSI Code for Red Text
#         color_code = "\033[91m"
#     else:
#         label = "Clean/Non-Hate"
#         # ANSI Code for Green Text
#         color_code = "\033[92m"

#     reset_code = "\033[0m"

#     # Print in the exact format you requested
#     print(f"Input Text: {text}")
#     print(f"{color_code}Predicted Label: {label}{reset_code}")


#     print("-" * 30)

# # --- TEST CASES ---
# examples = [
#     "You are an amazing person, keep it up!",
#     "I hate you, you are stupid and ugly.",
#     "The weather is very nice today.",
#     "Go back to your own country you loser."
# ]

# print("--- System Test Results ---\n")
# for ex in examples:
#     predict_status(ex)

# --------------------------------------------------------------


def predict_status(text):
    # Clean input
    cleaned_input = clean_text(text)

    # Model prediction
    prediction = model_pipeline.predict([cleaned_input])[0]
    prob = model_pipeline.predict_proba([cleaned_input])[0][1]

    # Extract words from text
    words = re.findall(r'\b\w+\b', cleaned_input.lower())

    # Find hate words present
    found_words = sorted(set(
        word for word in words
        if word in hate_words
    ))

    # Label
    if prediction == 1:
        label = "Hate/Toxic"
        color_code = "\033[91m"  # Red
    else:
        label = "Clean/Non-Hate"
        color_code = "\033[92m"  # Green

    reset_code = "\033[0m"

    print(f"Input Text: {text}")
    print(f"{color_code}Predicted Label: {label}{reset_code}")
    print(f"Confidence: {prob:.2%}")

    if found_words:
        print(f"Toxic Words Found: {', '.join(found_words)}")
    else:
        print("Toxic Words Found: None")

    print("-" * 50)


# ---------------------------------
# TEST CASES
# ---------------------------------

examples = [
    "You are an amazing person, keep it up!",
    "I hate you, you are stupid and ugly.",
    "The weather is very nice today.",
    "Go back to your own country you loser.",
    "You are a disgusting idiot.",
]

print("\n--- System Test Results ---\n")

for text in examples:
    predict_status(text)

***Cell 1: Imports & Setup***

***Logistic Regression***

In [ ]:
# Install necessary libraries
!pip install seaborn matplotlib scikit-learn pandas numpy joblib
!pip install datasets pandas
import os
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_curve, auc

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

***Cell 2: Load Data (Robust Version)***

***Hate Words Load Data***

In [ ]:
from datasets import load_dataset

# Load dataset
dataset = load_dataset("CyberTeensLLM/hate-normal-words")

# Check available splits
print(dataset)

# Convert all words into a set for fast lookup
hate_words = set()

for split in dataset.keys():
    df = dataset[split].to_pandas()

    # Replace 'word' with the actual column name after inspecting df.columns
    for word in df.iloc[:, 0]:
        if isinstance(word, str):
            hate_words.add(word.lower())

print(f"Loaded {len(hate_words)} hate words")

In [ ]:
# Define your path (Ensure this matches your Drive folder structure)
DATA_DIR = "/content/drive/MyDrive/NLP Project College/Dataset"

def load_and_merge_data(base_path):
    # List of files to process
    files = [
        ('aggression_parsed_dataset.csv', 'aggression'),
        ('attack_parsed_dataset.csv', 'attack'),
        ('toxicity_parsed_dataset.csv', 'toxicity'),
        ('kaggle_parsed_dataset.csv', 'kaggle'),
        ('twitter_parsed_dataset.csv', 'twitter'),
        ('twitter_racism_parsed_dataset.csv', 'twitter_racism'),
        ('twitter_sexism_parsed_dataset.csv', 'twitter_sexism'),
        ('youtube_parsed_dataset.csv', 'youtube')
    ]

    dfs = []
    if not os.path.exists(base_path):
        print(f"Error: Directory not found at {base_path}")
        return pd.DataFrame() # Return empty if path wrong

    for filename, source in files:
        path = os.path.join(base_path, filename)
        if os.path.exists(path):
            print(f"Loading {filename}...")
            try:
                # Read CSV (handling potential bad lines)
                temp_df = pd.read_csv(path, on_bad_lines='skip')

                # Standardize columns
                if "Text" in temp_df.columns and "oh_label" in temp_df.columns:
                    temp_df = temp_df[["Text", "oh_label"]].copy()
                    temp_df["source"] = source
                    dfs.append(temp_df)
            except Exception as e:
                print(f"Error loading {filename}: {e}")

    if dfs:
        full_df = pd.concat(dfs, ignore_index=True)
        return full_df
    else:
        return pd.DataFrame()

# Load Data
df = load_and_merge_data(DATA_DIR)

# Basic Cleanup
if not df.empty:
    df.dropna(subset=['Text', 'oh_label'], inplace=True)
    df['oh_label'] = df['oh_label'].astype(int)
    print(f"\nTotal Data Points: {df.shape[0]}")
    print("Class Distribution:\n", df['oh_label'].value_counts())
else:
    print("No data loaded. Please check your Google Drive path.")

***Cell 3: Pipeline Training***

In [ ]:
def clean_text(text):
    text = str(text).lower() # Convert to lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text) # Remove special characters
    text = re.sub(r'\s+', ' ', text).strip() # Remove extra spaces
    return text

# Apply cleaning
df['clean_text'] = df['Text'].apply(clean_text)

X = df['clean_text']
y = df['oh_label']

# Split Data (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Create Pipeline
# We use Class Weight 'balanced' to handle any imbalance in the dataset automatically
model_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=50000, ngram_range=(1, 3))),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', n_jobs=-1))
])

print("Training Model (this may take a minute)...")
model_pipeline.fit(X_train, y_train)
print("Training Complete.")

***Cell 4: Evaluation Metrics***

In [ ]:
# # Predictions
# y_pred = model_pipeline.predict(X_test)

# # Classification Report
# print("\n--- Classification Report ---\n")
# print(classification_report(y_test, y_pred))

# # Confusion Matrix Heatmap
# plt.figure(figsize=(6, 5))
# cm = confusion_matrix(y_test, y_pred)
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
#             xticklabels=['Clean', 'Hate'],
#             yticklabels=['Clean', 'Hate'])
# plt.xlabel('Predicted')
# plt.ylabel('Actual')
# plt.title('Confusion Matrix')
# plt.show()

# ---------------------------------------------------------------

# # Predictions
# y_pred = model_pipeline.predict(X_test)

# # Get predicted probabilities for the positive class (class 1)
# y_prob = model_pipeline.predict_proba(X_test)[:, 1]

# # Classification Report
# print("\n--- Classification Report ---\n")
# print(classification_report(y_test, y_pred))

# # Confusion Matrix Heatmap
# plt.figure(figsize=(6, 5))
# cm = confusion_matrix(y_test, y_pred)
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
#             xticklabels=['Clean', 'Hate'],
#             yticklabels=['Clean', 'Hate'])
# plt.xlabel('Predicted')
# plt.ylabel('Actual')
# plt.title('Confusion Matrix')
# plt.show()

# # ROC Curve (Measure of how well it separates classes)
# fpr, tpr, _ = roc_curve(y_test, y_prob)
# roc_auc = auc(fpr, tpr)

# plt.figure(figsize=(6, 5))
# plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
# plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
# plt.xlabel('False Positive Rate')
# plt.ylabel('True Positive Rate')
# plt.title('Receiver Operating Characteristic (ROC)')
# plt.legend(loc="lower right")
# plt.show()




# ----------------------------------------------------------------


# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

# Create side-by-side plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# -------------------------
# Confusion Matrix
# -------------------------
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Clean', 'Hate'],
    yticklabels=['Clean', 'Hate'],
    ax=axes[0]
)

axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# -------------------------
# ROC Curve
# -------------------------
axes[1].plot(
    fpr,
    tpr,
    color='darkorange',
    lw=2,
    label=f'ROC curve (AUC = {roc_auc:.4f})'
)

axes[1].plot(
    [0, 1],
    [0, 1],
    color='navy',
    lw=2,
    linestyle='--'
)

axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

***Accuracy***

In [ ]:
# Accuracy ==> Training
train_acc = model_pipeline.score(X_train, y_train)

# Accuracy ==> Test
test_acc = model_pipeline.score(X_test, y_test)

print(f"Training Accuracy: {train_acc * 100:.2f}%")
print(f"Testing Accuracy:  {test_acc * 100:.2f}%")

***Cell 5: The Specific Output You Requested [UPDATED]***

In [ ]:
# def predict_status(text):
#     # Clean the input text
#     cleaned_input = clean_text(text)

#     # Predict (returns 0 or 1)
#     prediction = model_pipeline.predict([cleaned_input])[0]

#     # Get Probability (confidence)
#     prob = model_pipeline.predict_proba([cleaned_input])[0][1]

#     # LOGIC: 0 = Clean, 1 = Hate
#     if prediction == 1:
#         label = "Hate/Toxic"
#         # ANSI Code for Red Text
#         color_code = "\033[91m"
#     else:
#         label = "Clean/Non-Hate"
#         # ANSI Code for Green Text
#         color_code = "\033[92m"

#     reset_code = "\033[0m"

#     # Print in the exact format you requested
#     print(f"Input Text: {text}")
#     print(f"{color_code}Predicted Label: {label}{reset_code}")


#     print("-" * 30)

# # --- TEST CASES ---
# examples = [
#     "You are an amazing person, keep it up!",
#     "I hate you, you are stupid and ugly.",
#     "The weather is very nice today.",
#     "Go back to your own country you loser."
# ]

# print("--- System Test Results ---\n")
# for ex in examples:
#     predict_status(ex)

# --------------------------------------------------------------


def predict_status(text):
    # Clean input
    cleaned_input = clean_text(text)

    # Model prediction
    prediction = model_pipeline.predict([cleaned_input])[0]
    prob = model_pipeline.predict_proba([cleaned_input])[0][1]

    # Extract words from text
    words = re.findall(r'\b\w+\b', cleaned_input.lower())

    # Find hate words present
    found_words = sorted(set(
        word for word in words
        if word in hate_words
    ))

    # Label
    if prediction == 1:
        label = "Hate/Toxic"
        color_code = "\033[91m"  # Red
    else:
        label = "Clean/Non-Hate"
        color_code = "\033[92m"  # Green

    reset_code = "\033[0m"

    print(f"Input Text: {text}")
    print(f"{color_code}Predicted Label: {label}{reset_code}")
    print(f"Confidence: {prob:.2%}")

    if found_words:
        print(f"Toxic Words Found: {', '.join(found_words)}")
    else:
        print("Toxic Words Found: None")

    print("-" * 50)


# ---------------------------------
# TEST CASES
# ---------------------------------

examples = [
    "You are an amazing person, keep it up!",
    "I hate you, you are stupid and ugly.",
    "The weather is very nice today.",
    "Go back to your own country you loser.",
    "You are a disgusting idiot.",
]

print("\n--- System Test Results ---\n")

for text in examples:
    predict_status(text)

***Cell 6: Interactive Testing (Optional)***

In [ ]:
print("--- Live Hate Speech Detection ---")
print("Type 'exit' to stop.")

while True:
    user_input = input("\nEnter text: ")
    if user_input.lower() == 'exit':
        break
    predict_status(user_input)

***Cell 7: Save the Model***

In [ ]:
save_path = os.path.join(DATA_DIR, "final_hate_speech_model.joblib")
joblib.dump(model_pipeline, save_path)
print(f"Model saved successfully to: {save_path}")